## Inicialización

### Configurar y montar Google Drive

In [7]:
import sys
from google.colab import drive

drive.mount('/content/drive')

ruta_base = '/content/drive/MyDrive/3.Hiruhilekoa/RETO3-SistemaGestiónMicrored/objetivo1/'

# Añadir la ruta al sistema
sys.path.append(ruta_base)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Importación de librerías

In [2]:
!pip install -U pymgrid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.1 MB/s eta 0:00:00
  Created wheel for pymgrid: filename=pymgrid-1.2.2-py3-none-any.whl size=3492850 sha256=5524c0d27a323749ae253cad1f95e8a25739ec0a9569f30be71c3183951e5c81
  Stored in directory: /root/.cache/pip/wheels/22/74/ee/35d2b7ad23381dc521b2c8670c6e30be0b2e9fe80a1fe03160
Successfully built pymgrid


In [8]:
import pandas as pd
import numpy as np
from pymgrid import Microgrid
from pymgrid.modules import GridModule, BatteryModule, LoadModule, RenewableModule

from custom_env_tabular import CustomEnvTabular

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Carga y preparación de datos

In [10]:
# 2.1 PRECIOS DE LA RED (Península 2025)
ruta_precios = ruta_base + 'data/precio2025-peninsula.csv'
df_precios = pd.read_csv(ruta_precios, sep=';')
df_precios['datetime'] = pd.to_datetime(df_precios['datetime'])
df_precios = df_precios.sort_values('datetime').reset_index(drop=True)
# Convertir de €/MWh a €/kWh
precios_kwh = df_precios['value'].values / 1000.0

# 2.2 DEMANDA (LOAD) Y GENERACIÓN SOLAR (PV)
ruta_load = ruta_base + 'data/pymgrid_data/load/RefBldgFullServiceRestaurantNew2004_v1.3_7.1_6A_USA_MN_MINNEAPOLIS.csv'
ruta_pv = ruta_base + 'data/pymgrid_data/pv/SanFrancisco_724940TYA.csv'

df_load = pd.read_csv(ruta_load)
df_pv = pd.read_csv(ruta_pv)

# Extraer los valores numéricos.
# Si las columnas no se llaman 'valor', pillamos la última columna asumiendo que es el dato de kW.
try:
    load_series = df_load['valor'].values
    pv_series = df_pv['valor'].values
except KeyError:
    print("Aviso: Ajustando la lectura de columnas para Load y PV...")
    load_series = df_load.iloc[:, -1].values
    pv_series = df_pv.iloc[:, -1].values

# Asegurarnos de que todas las series tienen la misma longitud (evitar desajustes si un año es bisiesto, etc.)
min_len = min(len(precios_kwh), len(load_series), len(pv_series), 8760)
precios_kwh = precios_kwh[:min_len]
load_series = load_series[:min_len]
pv_series = pv_series[:min_len]

/tmp/ipykernel_8107/2810487095.py:4: FutureWarning:

In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



Aviso: Ajustando la lectura de columnas para Load y PV...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



### Creación de los módulos de pymgrid

In [15]:
# RED ELÉCTRICA (GridModule)
# En las versiones recientes de pymgrid, los datos de la red se pasan como un DataFrame
grid_ts = pd.DataFrame({
    'import_price': precios_kwh,
    'export_price': precios_kwh * 0.5,
    'co2_per_kwh': 0.0                 # Rellenamos con 0, pero pymgrid necesita esta columna
})

grid = GridModule(
    max_import=10000.0,
    max_export=10000.0,
    time_series=grid_ts
)

# BATERÍA (BatteryModule)
battery = BatteryModule(
    min_capacity=10.0,
    max_capacity=200.0,
    max_charge=50.0,
    max_discharge=50.0,
    efficiency=0.9,
    init_soc=0.5
)

# DEMANDA Y PLACAS SOLARES
# Les pasamos directamente los arrays con los kW de cada hora
load = LoadModule(time_series=load_series)
pv = RenewableModule(time_series=pv_series)

### Ensamblaje de la microrred y el entorno gymnasium

In [16]:
# Creamos la lista de tuplas con el nombre ('string') y el objeto de cada módulo
modules = [
    ('grid', grid),
    ('battery', battery),
    ('load', load),
    ('pv', pv)
]

# Construimos la microrred
microrred = Microgrid(modules)

# La conectamos a vuestro entorno
env = CustomEnvTabular(
    pymgrid_network=microrred,
    horizon=min_len,
    low_soc_penalty=10.0
)

### Prueba de funcionamiento

In [19]:
obs, info = env.reset()
print("¡Entorno inicializado con éxito, sin errores!")
print(f"Estado Inicial Discretizado [Carga_Neta, Batería_SoC, Precio, Hora]: {obs}")
print(f"Precio extraído para la primera hora: {info['current_import_price']:.4f} €/kWh")

¡Entorno inicializado con éxito, sin errores!
Estado Inicial Discretizado [Carga_Neta, Batería_SoC, Precio, Hora]: [2 5 3 0]
Precio extraído para la primera hora: 0.1832 €/kWh
